# 🚀 Fine-Tune LLM with Azure AI Foundry + CI/CD on AWS
## End-to-End Single-File Project Notebook

> **Project:** Generate a synthetic Customer-Support dataset using **Groq LLaMA**, fine-tune **GPT-4o** on **Azure AI Foundry**, build a streaming React chat UI, and deploy via **AWS CodePipeline → S3**.

---
### 📂 Project Structure (mirrors GitHub repo)
```
project-root/
├── DATA JSONL FILES/
│   ├── train.jsonl
│   └── validation.jsonl
├── src/
│   ├── App.jsx
│   ├── App.css
│   ├── index.css
│   ├── main.jsx
│   └── components/
│       ├── ChatInput.jsx
│       ├── ChatMessages.jsx
│       ├── ConfigModal.jsx
│       ├── Sidebar.jsx
│       └── TopBar.jsx
├── server/
│   ├── index.js
│   └── package.json
├── public/
│   └── vite.svg
├── index.html
├── vite.config.js
├── package.json
├── buildspec.yml
└── .gitignore
```
| Item | Value |
|------|-------|
| LangChain | **1.2.0** (core + community + groq) |
| LLM | Groq `llama-3.1-8b-instant` / `llama3-70b-8192` |
| Secret | `GROQ_API_KEY` in Colab Secrets (🔑 panel) |


---
## 📦 Step 0 — Install All Python Dependencies


In [ ]:
# Step 0.1 — Install all required packages
# LangChain 1.2.0 (latest) — do NOT use 0.2.x
!pip install -q \
    langchain==1.2.0 \
    langchain-core==1.2.0 \
    langchain-community==1.2.0 \
    langchain-groq==0.3.2 \
    groq==0.28.0 \
    openai==1.82.1 \
    tiktoken==0.9.0 \
    datasets==3.6.0 \
    pandas==2.2.3 \
    numpy==2.2.6 \
    matplotlib==3.10.3 \
    seaborn==0.13.2 \
    tqdm==4.67.1 \
    jsonlines==4.0.0

print("All dependencies installed.")


---
## 🔐 Step 1 — Authentication & Configuration

Load `GROQ_API_KEY` from Colab Secrets (🔑 left sidebar). Optionally configure Azure credentials for the inference section.


In [ ]:
# Step 1.1 — Load GROQ_API_KEY from Colab Secrets
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is empty — add it via the Colab Secrets panel.")
    print("GROQ_API_KEY loaded from Colab Secrets.")
except ImportError:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    print("Running locally — reading GROQ_API_KEY from environment.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


In [ ]:
# Step 1.2 — Model & project configuration
# ─────────────────────────────────────────────────────────────────
# Groq model options (free tier):
#   llama-3.1-8b-instant   → fastest, ~6 000 TPM
#   llama3-70b-8192         → higher quality, ~14 400 req/day
#   llama-3.3-70b-versatile → latest versatile model
# ─────────────────────────────────────────────────────────────────
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"    # primary (speed + TPM budget)
GROQ_MODEL_HQ    = "llama3-70b-8192"          # high-quality alternative
MAX_TOKENS       = 1024
TEMPERATURE      = 0.7

# Azure OpenAI (fill in after fine-tuning — used in inference section)
AZURE_ENDPOINT    = ""   # https://<resource>.cognitiveservices.azure.com/
AZURE_API_KEY     = ""
AZURE_DEPLOYMENT  = ""   # gpt-4o-2024-08-06-project-demo
AZURE_API_VERSION = "2024-12-01-preview"

print(f"Primary model : {GROQ_MODEL_FAST}")
print(f"HQ model      : {GROQ_MODEL_HQ}")


---
## 🗂️ Step 2 — Create Project Directory Structure


In [ ]:
# Step 2.1 — Create all project directories (mirrors GitHub repo)
import pathlib

BASE = pathlib.Path("project-root")
for d in [
    BASE / "DATA JSONL FILES",
    BASE / "src" / "components",
    BASE / "src" / "assets",
    BASE / "server",
    BASE / "public",
]:
    d.mkdir(parents=True, exist_ok=True)

print("Directory tree created:")
for p in sorted(BASE.rglob("*")):
    if p.is_dir():
        depth  = len(p.relative_to(BASE).parts)
        print("    " * depth + f"📁 {p.name}/")


---
## 🧠 Step 3 — LangChain 1.2.0 Setup

Using the latest LangChain 1.2.0 import paths (not 0.2.x).


In [ ]:
# Step 3.1 — LangChain 1.2.0 imports (all from langchain_core / langchain_groq)
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages      import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables      import RunnablePassthrough, RunnableLambda
from langchain_groq                import ChatGroq

print("LangChain 1.2.0 imports OK.")


In [ ]:
# Step 3.2 — Initialise Groq LLM clients
llm_fast = ChatGroq(
    model=GROQ_MODEL_FAST,
    api_key=GROQ_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)
llm_hq = ChatGroq(
    model=GROQ_MODEL_HQ,
    api_key=GROQ_API_KEY,
    temperature=0.8,
    max_tokens=MAX_TOKENS,
)

# Step 3.3 — System prompts
SYSTEM_PROMPT = (
    "You are a professional and friendly customer support assistant. "
    "You are knowledgeable, empathetic, and always aim to resolve customer "
    "issues efficiently. Provide clear, concise, and helpful responses. "
    "If you cannot resolve an issue, politely guide the customer to "
    "escalate to a human agent."
)
MENTOR_SYSTEM_PROMPT = (
    "You are an AI Project Mentor for an educational platform. "
    "Help learners with project recommendations, skill-gap analysis, "
    "roadmap planning, architecture design, and MLOps deployment guidance."
)
print("LLM clients and system prompts ready.")


---
## 📊 Step 4 — Synthetic Dataset Generation

Generate **400–600 examples** in Azure OpenAI fine-tuning JSONL format.

Each line:
```json
{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
```


In [ ]:
# Step 4.1 — Topic taxonomy for dataset diversity
import random

TOPIC_CATEGORIES = {
    "order_tracking": [
        "Where is my order?",
        "My package shows delivered but I haven't received it.",
        "How do I track my order in real time?",
        "My tracking number doesn't work.",
        "How long does standard shipping take?",
    ],
    "returns_refunds": [
        "What is your return policy?",
        "How do I return a damaged item?",
        "When will I receive my refund?",
        "Can I exchange an item instead of returning it?",
        "I returned my order 2 weeks ago and haven't gotten a refund.",
    ],
    "account_issues": [
        "I forgot my password.",
        "How do I change my email address?",
        "My account has been locked.",
        "How do I delete my account?",
        "I'm being charged twice for the same order.",
    ],
    "product_questions": [
        "Does this product come with a warranty?",
        "What are the dimensions of the product?",
        "Is this item compatible with my device?",
        "What materials is this product made of?",
        "Can I get a bulk discount?",
    ],
    "technical_support": [
        "The app keeps crashing on my phone.",
        "I can't complete my checkout — it keeps failing.",
        "The promo code isn't working.",
        "I'm not receiving order confirmation emails.",
        "The website shows an error when I log in.",
    ],
    "escalation": [
        "I've been waiting 3 weeks and no one is helping me.",
        "This is the third time I'm contacting support for the same issue.",
        "I want to speak to a manager.",
        "Your automated responses are not helping my situation.",
    ],
    "ai_project_mentor": [
        "What ML projects should a beginner start with?",
        "How do I go from data analyst to ML engineer?",
        "Can you recommend an end-to-end NLP project?",
        "How should I structure my MLOps portfolio?",
        "What is the difference between fine-tuning and RAG?",
    ],
}

total_seed = sum(len(v) for v in TOPIC_CATEGORIES.values())
print(f"Defined {total_seed} seed topics across {len(TOPIC_CATEGORIES)} categories.")


In [ ]:
# Step 4.2 — LangChain 1.2.0 generation chain (LCEL pipe syntax)
import json, re

generation_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a dataset generation expert. Generate realistic customer "
        "support dialogues in strict JSON. Output ONLY valid JSON — no "
        "markdown fences, no preamble."
    ),
    HumanMessagePromptTemplate.from_template(
        'Generate a customer support conversation for: "{topic}" (category: {category}, complexity: {complexity})\n'
        "Return ONLY this JSON:\n"
        "{{"
        '"user_query": "<realistic user message>", '
        '"assistant_response": "<professional empathetic response>", '
        '"follow_up_user": "<user follow-up>", '
        '"follow_up_assistant": "<assistant follow-up>"'
        "}}"
    ),
])

# LangChain 1.2.0 LCEL chain
generation_chain = generation_prompt | llm_fast | StrOutputParser()

def parse_json_safe(text: str):
    text = re.sub(r"```(?:json)?", "", text).strip().rstrip("`").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except Exception:
                pass
    return None

print("LangChain 1.2.0 generation chain built — type:", type(generation_chain).__name__)


In [ ]:
# Step 4.3 — Generate training dataset (400 examples)
# Rate-limit safe: sleeps between calls.
# Free Groq tier: ~6 000 TPM for llama-3.1-8b-instant.
import time
from tqdm.auto import tqdm

TRAIN_SIZE    = 400
SLEEP_BETWEEN = 1.5   # seconds — adjust up if rate-limited

all_topics = [(cat, topic) for cat, topics in TOPIC_CATEGORIES.items() for topic in topics]
complexities = ["beginner", "intermediate", "advanced"]

random.seed(42)
pool = (all_topics * ((TRAIN_SIZE // len(all_topics)) + 2))
random.shuffle(pool)
pool = pool[:TRAIN_SIZE]

train_examples, skipped = [], 0

for i, (cat, topic) in enumerate(tqdm(pool, desc="Generating train data")):
    try:
        raw    = generation_chain.invoke({"topic": topic, "category": cat, "complexity": complexities[i % 3]})
        parsed = parse_json_safe(raw)
        if not parsed:
            skipped += 1
            continue
        ex = {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": parsed.get("user_query", topic)},
                {"role": "assistant", "content": parsed.get("assistant_response", "")},
                {"role": "user",      "content": parsed.get("follow_up_user", "")},
                {"role": "assistant", "content": parsed.get("follow_up_assistant", "")},
            ]
        }
        ex["messages"] = [m for m in ex["messages"] if m["content"].strip()]
        train_examples.append(ex)
        time.sleep(SLEEP_BETWEEN)
    except Exception as e:
        skipped += 1
        if "rate" in str(e).lower():
            print(f"Rate limit at {i} — sleeping 30 s...")
            time.sleep(30)

print(f"Generated {len(train_examples)} training examples ({skipped} skipped).")


In [ ]:
# Step 4.4 — Generate validation dataset (50 examples)
VAL_SIZE = 50

random.seed(99)
val_pool = random.sample(pool, min(VAL_SIZE, len(pool)))
val_examples = []

for i, (cat, topic) in enumerate(tqdm(val_pool, desc="Generating val data")):
    try:
        raw    = generation_chain.invoke({"topic": topic, "category": cat, "complexity": complexities[(i + 1) % 3]})
        parsed = parse_json_safe(raw)
        if not parsed:
            continue
        ex = {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": parsed.get("user_query", topic)},
                {"role": "assistant", "content": parsed.get("assistant_response", "")},
            ]
        }
        ex["messages"] = [m for m in ex["messages"] if m["content"].strip()]
        val_examples.append(ex)
        time.sleep(SLEEP_BETWEEN)
    except Exception as e:
        if "rate" in str(e).lower():
            time.sleep(30)

print(f"Generated {len(val_examples)} validation examples.")


In [ ]:
# Step 4.5 — Save JSONL files
import jsonlines

TRAIN_PATH = BASE / "DATA JSONL FILES" / "train.jsonl"
VAL_PATH   = BASE / "DATA JSONL FILES" / "validation.jsonl"

with jsonlines.open(TRAIN_PATH, mode="w") as w:
    for ex in train_examples:
        w.write(ex)

with jsonlines.open(VAL_PATH, mode="w") as w:
    for ex in val_examples:
        w.write(ex)

print(f"train.jsonl      → {TRAIN_PATH}  ({len(train_examples)} examples)")
print(f"validation.jsonl → {VAL_PATH}  ({len(val_examples)} examples)")


---
## 🔍 Step 5 — Dataset Validation & Statistics


In [ ]:
# Step 5.1 — Validate JSONL against Azure OpenAI fine-tuning format
import tiktoken, jsonlines, pandas as pd

def validate_jsonl(path, label):
    enc = tiktoken.get_encoding("cl100k_base")
    examples, errors, tokens, turns = [], [], [], []
    with jsonlines.open(path) as r:
        for i, obj in enumerate(r):
            if "messages" not in obj:
                errors.append(f"Row {i}: missing 'messages'"); continue
            msgs  = obj["messages"]
            roles = [m.get("role") for m in msgs]
            if "system"    not in roles: errors.append(f"Row {i}: no system msg")
            if "user"      not in roles: errors.append(f"Row {i}: no user msg")
            if "assistant" not in roles: errors.append(f"Row {i}: no assistant msg")
            tc = sum(len(enc.encode(m.get("content", ""))) for m in msgs)
            tokens.append(tc); turns.append(len(msgs)); examples.append(obj)
    if errors:
        print(f"WARNINGS in {label}:")
        for e in errors[:5]: print("  ", e)
    else:
        print(f"{label}: no format errors.")
    return {
        "label": label, "total_examples": len(examples),
        "errors": len(errors),
        "avg_tokens": round(sum(tokens) / max(len(tokens), 1), 1),
        "max_tokens": max(tokens) if tokens else 0,
        "avg_turns":  round(sum(turns)  / max(len(turns),  1), 1),
    }

ts = validate_jsonl(TRAIN_PATH, "train.jsonl")
vs = validate_jsonl(VAL_PATH,   "validation.jsonl")
print()
print(pd.DataFrame([ts, vs]).to_string(index=False))


In [ ]:
# Step 5.2 — Token distribution plot
import matplotlib.pyplot as plt
import tiktoken, jsonlines

enc = tiktoken.get_encoding("cl100k_base")

def token_counts(path):
    with jsonlines.open(path) as r:
        return [sum(len(enc.encode(m.get("content",""))) for m in obj.get("messages",[])) for obj in r]

tc_train = token_counts(TRAIN_PATH)
tc_val   = token_counts(VAL_PATH)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Token Distribution per Example", fontsize=14, fontweight="bold")

ax1.hist(tc_train, bins=30, color="#6366f1", edgecolor="white", alpha=0.85)
ax1.set_title("Training Set"); ax1.set_xlabel("Tokens"); ax1.set_ylabel("Count")
ax1.axvline(4096, color="red", linestyle="--", label="4096 limit"); ax1.legend()

ax2.hist(tc_val, bins=20, color="#8b5cf6", edgecolor="white", alpha=0.85)
ax2.set_title("Validation Set"); ax2.set_xlabel("Tokens")
ax2.axvline(4096, color="red", linestyle="--", label="4096 limit"); ax2.legend()

plt.tight_layout()
plt.savefig(BASE / "DATA JSONL FILES" / "token_distribution.png", dpi=120)
plt.show()
print("Token distribution chart saved.")


---
## 💾 Step 6 — Write All Project Source Files

Every file from the GitHub repo written to `project-root/` verbatim.


In [ ]:
# Step 6.1 — vite.config.js
(BASE / "vite.config.js").write_text(
    "import { defineConfig } from 'vite'\n"
    "import react from '@vitejs/plugin-react'\n\n"
    "export default defineConfig({\n"
    "  plugins: [react()],\n"
    "  server: { port: 5173 },\n"
    "})\n"
)
print("vite.config.js written.")


In [ ]:
# Step 6.2 — package.json
import json as _json
pkg = {
    "name": "support-bot-ai", "private": True, "version": "1.0.0", "type": "module",
    "scripts": {"dev": "vite", "server": "node server/index.js", "build": "vite build",
                "lint": "eslint .", "preview": "vite preview"},
    "dependencies": {"react": "^19.2.0", "react-dom": "^19.2.0"},
    "devDependencies": {
        "@eslint/js": "^9.39.1", "@types/react": "^19.2.7",
        "@types/react-dom": "^19.2.3", "@vitejs/plugin-react": "^5.1.1",
        "eslint": "^9.39.1", "eslint-plugin-react-hooks": "^7.0.1",
        "eslint-plugin-react-refresh": "^0.4.24", "globals": "^16.5.0",
        "vite": "^7.3.1",
    },
}
(BASE / "package.json").write_text(_json.dumps(pkg, indent=2))
print("package.json written.")


In [ ]:
# Step 6.3 — buildspec.yml  (AWS CodeBuild spec)
buildspec = [
    "version: 0.2",
    "",
    "phases:",
    "  install:",
    "    runtime-versions:",
    "      nodejs: 18",
    "    commands:",
    "      - echo Installing dependencies...",
    "      - npm ci --legacy-peer-deps",
    "",
    "  build:",
    "    commands:",
    "      - echo Building the React app...",
    "      - npm run build",
    "",
    "  post_build:",
    "    commands:",
    "      - echo Build complete. Artifacts ready for S3 deployment.",
    "",
    "artifacts:",
    "  files:",
    "    - '**/*'",
    "  base-directory: dist",
    "  discard-paths: no",
    "",
]
(BASE / "buildspec.yml").write_text("\n".join(buildspec))
print("buildspec.yml written.")


In [ ]:
# Step 6.4 — .gitignore
lines = [
    "node_modules/", ".pnp", ".pnp.js",
    "dist/", "build/",
    ".env", ".env.local", ".env.*.local",
    "npm-debug.log*", "yarn-debug.log*", "yarn-error.log*",
    ".idea/", ".vscode/", "*.swp", "*.swo",
    ".DS_Store", "Thumbs.db",
    "__pycache__/", "*.py[cod]", ".ipynb_checkpoints/",
]
(BASE / ".gitignore").write_text("\n".join(lines) + "\n")
print(".gitignore written.")


In [ ]:
# Step 6.5 — index.html
html_lines = [
    "<!doctype html>",
    '<html lang="en">',
    "<head>",
    '  <meta charset="UTF-8" />',
    '  <link rel="icon" type="image/svg+xml"',
    "    href="data:image/svg+xml,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 100 100'><text y='.9em' font-size='90'>\U0001f916</text></svg>" />",
    '  <meta name="viewport" content="width=device-width, initial-scale=1.0" />',
    '  <meta name="description" content="AI-powered customer support chatbot built on Azure OpenAI GPT-4o" />',
    "  <title>KrishNaik Sir Website Bot</title>",
    "</head>",
    "<body>",
    '  <div id="root"></div>',
    '  <script type="module" src="/src/main.jsx"></script>',
    "</body>",
    "</html>",
    "",
]
(BASE / "index.html").write_text("\n".join(html_lines))
print("index.html written.")


In [ ]:
# Step 6.6 — src/main.jsx
main_jsx_lines = [
    "import { StrictMode } from 'react'",
    "import { createRoot } from 'react-dom/client'",
    "import './index.css'",
    "import App from './App.jsx'",
    "",
    "createRoot(document.getElementById('root')).render(",
    "  <StrictMode>",
    "    <App />",
    "  </StrictMode>,",
    ")",
    "",
]
(BASE / "src" / "main.jsx").write_text("\n".join(main_jsx_lines))
print("src/main.jsx written.")


In [ ]:
# Step 6.7 — src/App.css
app_css_parts = [
    "#root {",
    "  max-width: 1280px;",
    "  margin: 0 auto;",
    "  padding: 2rem;",
    "  text-align: center;",
    "}",
    ".logo {",
    "  height: 6em;",
    "  padding: 1.5em;",
    "  will-change: filter;",
    "  transition: filter 300ms;",
    "}",
    ".logo:hover { filter: drop-shadow(0 0 2em #646cffaa); }",
    ".logo.react:hover { filter: drop-shadow(0 0 2em #61dafbaa); }",
    "@keyframes logo-spin {",
    "  from { transform: rotate(0deg); }",
    "  to   { transform: rotate(360deg); }",
    "}",
    "@media (prefers-reduced-motion: no-preference) {",
    "  a:nth-of-type(2) .logo { animation: logo-spin infinite 20s linear; }",
    "}",
    ".card { padding: 2em; }",
    ".read-the-docs { color: #888; }",
    "",
]
(BASE / "src" / "App.css").write_text("\n".join(app_css_parts))
print("src/App.css written.")


In [ ]:
# Step 6.8 — src/index.css  (full design-system — dark theme)
# Written as a joined list to avoid Python parser conflicts with CSS values.
css = "\n".join([
    "@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');",
    ":root {",
    "  --bg-primary: #0a0a0f;",
    "  --bg-secondary: #12121a;",
    "  --bg-card: #1a1a28;",
    "  --bg-glass: rgba(255,255,255,0.04);",
    "  --bg-glass-hover: rgba(255,255,255,0.07);",
    "  --bg-input: rgba(255,255,255,0.06);",
    "  --bg-user-msg: linear-gradient(135deg,#6366f1 0%,#8b5cf6 100%);",
    "  --bg-bot-msg: rgba(255,255,255,0.05);",
    "  --accent-primary: #6366f1;",
    "  --accent-secondary: #8b5cf6;",
    "  --accent-glow: rgba(99,102,241,0.3);",
    "  --text-primary: #f1f5f9;",
    "  --text-secondary: #94a3b8;",
    "  --text-muted: #475569;",
    "  --text-accent: #818cf8;",
    "  --border-subtle: rgba(255,255,255,0.06);",
    "  --border-medium: rgba(255,255,255,0.1);",
    "  --border-accent: rgba(99,102,241,0.4);",
    "  --shadow-lg: 0 20px 60px rgba(0,0,0,0.6);",
    "  --radius-sm: 8px; --radius-md: 12px; --radius-lg: 16px;",
    "  --radius-xl: 24px; --radius-full: 9999px;",
    "  --transition: all 0.2s cubic-bezier(0.4,0,0.2,1);",
    "}",
    "*,*::before,*::after { box-sizing:border-box; margin:0; padding:0; }",
    "html,body,#root { height:100%; width:100%; }",
    "body { font-family:'Inter',sans-serif; background:var(--bg-primary); color:var(--text-primary); overflow:hidden; -webkit-font-smoothing:antialiased; }",
    ".app { display:flex; height:100vh; width:100vw; position:relative; overflow:hidden; }",
    ".app::before { content:''; position:fixed; inset:0; background: radial-gradient(ellipse 80% 50% at 20% 0%,rgba(99,102,241,0.12) 0%,transparent 60%), radial-gradient(ellipse 60% 40% at 80% 100%,rgba(139,92,246,0.1) 0%,transparent 60%); pointer-events:none; z-index:0; }",
    ".sidebar { width:280px; min-width:280px; background:var(--bg-secondary); border-right:1px solid var(--border-subtle); display:flex; flex-direction:column; position:relative; z-index:1; }",
    ".sidebar-header { padding:24px 20px 20px; border-bottom:1px solid var(--border-subtle); }",
    ".brand { display:flex; align-items:center; gap:12px; margin-bottom:20px; }",
    ".brand-icon { width:40px; height:40px; border-radius:var(--radius-md); background:var(--bg-user-msg); display:flex; align-items:center; justify-content:center; font-size:20px; box-shadow:0 4px 15px rgba(99,102,241,0.4); flex-shrink:0; }",
    ".brand-text h1 { font-size:16px; font-weight:700; color:var(--text-primary); letter-spacing:-0.3px; }",
    ".brand-text p { font-size:11px; color:var(--text-muted); letter-spacing:0.5px; text-transform:uppercase; }",
    ".new-chat-btn { width:100%; padding:10px 14px; border-radius:var(--radius-md); background:var(--bg-glass); border:1px solid var(--border-medium); color:var(--text-secondary); font-size:13px; font-weight:500; font-family:inherit; cursor:pointer; display:flex; align-items:center; gap:8px; transition:var(--transition); }",
    ".new-chat-btn:hover { background:var(--bg-glass-hover); color:var(--text-primary); border-color:var(--border-accent); }",
    ".new-chat-btn svg { width:16px; height:16px; flex-shrink:0; }",
    ".sidebar-content { flex:1; overflow-y:auto; padding:12px; }",
    ".sidebar-section-label { font-size:10px; font-weight:600; color:var(--text-muted); text-transform:uppercase; letter-spacing:1px; padding:0 8px 8px; }",
    ".conversation-item { padding:10px 12px; border-radius:var(--radius-sm); cursor:pointer; transition:var(--transition); margin-bottom:2px; display:flex; align-items:center; gap:10px; }",
    ".conversation-item:hover { background:var(--bg-glass); }",
    ".conversation-item.active { background:rgba(99,102,241,0.12); border:1px solid rgba(99,102,241,0.2); }",
    ".conversation-item-icon { font-size:14px; flex-shrink:0; }",
    ".conversation-item-text { font-size:13px; color:var(--text-secondary); overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }",
    ".conversation-item.active .conversation-item-text { color:var(--text-primary); }",
    ".sidebar-footer { padding:16px; border-top:1px solid var(--border-subtle); }",
    ".settings-btn { width:100%; padding:10px 14px; border-radius:var(--radius-md); background:transparent; border:1px solid var(--border-subtle); color:var(--text-muted); font-size:13px; font-family:inherit; cursor:pointer; display:flex; align-items:center; gap:8px; transition:var(--transition); }",
    ".settings-btn:hover { background:var(--bg-glass); color:var(--text-secondary); border-color:var(--border-medium); }",
    ".settings-btn svg { width:16px; height:16px; }",
    ".main-content { flex:1; display:flex; flex-direction:column; overflow:hidden; position:relative; z-index:1; }",
    ".topbar { display:flex; align-items:center; justify-content:space-between; padding:16px 24px; background:rgba(18,18,26,0.8); backdrop-filter:blur(20px); border-bottom:1px solid var(--border-subtle); flex-shrink:0; }",
    ".topbar-bot-info { display:flex; align-items:center; gap:12px; }",
    ".bot-avatar { width:40px; height:40px; border-radius:var(--radius-md); background:var(--bg-user-msg); display:flex; align-items:center; justify-content:center; font-size:20px; position:relative; box-shadow:0 4px 15px rgba(99,102,241,0.3); }",
    ".status-dot { position:absolute; bottom:-2px; right:-2px; width:10px; height:10px; border-radius:50%; background:#22c55e; border:2px solid var(--bg-secondary); animation:pulse-green 2s infinite; }",
    ".status-dot.offline { background:#f59e0b; animation:none; }",
    ".topbar-bot-info h2 { font-size:15px; font-weight:600; color:var(--text-primary); }",
    ".topbar-bot-info p { font-size:12px; color:var(--text-muted); }",
    ".topbar-bot-info p.offline-status { color:#f59e0b; }",
    ".topbar-actions { display:flex; align-items:center; gap:10px; }",
    ".icon-btn { width:36px; height:36px; border-radius:var(--radius-md); background:var(--bg-glass); border:1px solid var(--border-subtle); color:var(--text-muted); display:flex; align-items:center; justify-content:center; cursor:pointer; transition:var(--transition); }",
    ".icon-btn:hover { background:var(--bg-glass-hover); color:var(--text-secondary); border-color:var(--border-medium); }",
    ".icon-btn svg { width:16px; height:16px; }",
    ".messages-wrapper { flex:1; overflow-y:auto; position:relative; }",
    ".messages-container { max-width:780px; margin:0 auto; padding:24px 24px 16px; display:flex; flex-direction:column; gap:8px; min-height:100%; }",
    ".welcome-screen { flex:1; display:flex; flex-direction:column; align-items:center; justify-content:center; text-align:center; padding:40px 20px; gap:16px; }",
    ".welcome-icon { font-size:52px; filter:drop-shadow(0 8px 24px rgba(99,102,241,0.4)); }",
    ".welcome-screen h2 { font-size:22px; font-weight:700; color:var(--text-primary); letter-spacing:-0.3px; }",
    ".welcome-screen p { font-size:14px; color:var(--text-secondary); max-width:400px; line-height:1.6; }",
    ".welcome-chips { display:flex; flex-wrap:wrap; gap:8px; justify-content:center; margin-top:8px; }",
    ".welcome-chip { padding:8px 16px; border-radius:var(--radius-full); background:var(--bg-glass); border:1px solid var(--border-medium); color:var(--text-secondary); font-size:13px; font-family:inherit; cursor:pointer; transition:var(--transition); }",
    ".welcome-chip:hover { background:var(--bg-glass-hover); color:var(--text-primary); border-color:var(--border-accent); }",
    ".message-group { display:flex; gap:10px; animation:fadeInUp 0.3s ease; }",
    ".message-group.user { flex-direction:row-reverse; }",
    "@keyframes fadeInUp { from { opacity:0; transform:translateY(10px); } to { opacity:1; transform:translateY(0); } }",
    ".message-avatar { width:28px; height:28px; border-radius:var(--radius-sm); flex-shrink:0; display:flex; align-items:center; justify-content:center; font-size:14px; margin-top:2px; }",
    ".bot-avatar-sm { background:var(--bg-user-msg); box-shadow:0 2px 8px rgba(99,102,241,0.3); }",
    ".user-avatar-sm { background:var(--bg-glass); border:1px solid var(--border-medium); }",
    ".message-body { display:flex; flex-direction:column; gap:4px; max-width:72%; }",
    ".message-group.user .message-body { align-items:flex-end; }",
    ".message-bubble { padding:10px 14px; border-radius:var(--radius-lg); font-size:14px; line-height:1.6; word-wrap:break-word; }",
    ".message-bubble.bot { background:var(--bg-bot-msg); border:1px solid var(--border-subtle); color:var(--text-primary); border-top-left-radius:4px; }",
    ".message-bubble.user { background:var(--bg-user-msg); color:white; border-top-right-radius:4px; box-shadow:0 4px 15px rgba(99,102,241,0.3); }",
    ".message-bubble.error-msg { border-color:rgba(239,68,68,0.3); background:rgba(239,68,68,0.08); }",
    ".message-time { font-size:10px; color:var(--text-muted); padding:0 4px; }",
    ".typing-bubble { display:flex; gap:5px; padding:12px 16px; background:var(--bg-bot-msg); border:1px solid var(--border-subtle); border-radius:var(--radius-lg); border-top-left-radius:4px; }",
    ".typing-dot { width:7px; height:7px; border-radius:50%; background:var(--text-muted); animation:typingDot 1.4s ease-in-out infinite; }",
    ".typing-dot:nth-child(2) { animation-delay:0.2s; }",
    ".typing-dot:nth-child(3) { animation-delay:0.4s; }",
    "@keyframes typingDot { 0%,60%,100% { opacity:0.3; transform:scale(1); } 30% { opacity:1; transform:scale(1.2); } }",
    ".input-area { padding:16px 24px 20px; background:rgba(10,10,15,0.6); backdrop-filter:blur(20px); border-top:1px solid var(--border-subtle); }",
    ".input-container { max-width:780px; margin:0 auto; }",
    ".input-box { display:flex; align-items:flex-end; gap:12px; background:var(--bg-input); border:1px solid var(--border-medium); border-radius:var(--radius-xl); padding:12px 16px; transition:var(--transition); }",
    ".input-box:focus-within { border-color:var(--border-accent); box-shadow:0 0 0 3px var(--accent-glow); }",
    ".input-box textarea { flex:1; background:transparent; border:none; outline:none; color:var(--text-primary); font-family:inherit; font-size:14px; line-height:1.5; resize:none; max-height:140px; overflow-y:auto; padding:2px 0; }",
    ".input-box textarea::placeholder { color:var(--text-muted); }",
    ".send-btn { width:36px; height:36px; border-radius:var(--radius-md); background:linear-gradient(135deg,#6366f1,#8b5cf6); border:none; color:white; display:flex; align-items:center; justify-content:center; cursor:pointer; flex-shrink:0; transition:var(--transition); box-shadow:0 2px 10px rgba(99,102,241,0.4); }",
    ".send-btn:hover:not(:disabled) { transform:translateY(-1px); box-shadow:0 6px 20px rgba(99,102,241,0.5); }",
    ".send-btn:disabled { opacity:0.4; cursor:not-allowed; transform:none; }",
    ".send-btn svg { width:16px; height:16px; }",
    ".input-footer { text-align:center; margin-top:10px; font-size:11px; color:var(--text-muted); }",
    ".modal-overlay { position:fixed; inset:0; background:rgba(0,0,0,0.75); backdrop-filter:blur(8px); display:flex; align-items:center; justify-content:center; z-index:100; animation:fadeIn 0.2s ease; padding:20px; }",
    "@keyframes fadeIn { from{opacity:0;} to{opacity:1;} }",
    ".modal { background:var(--bg-card); border:1px solid var(--border-medium); border-radius:var(--radius-xl); width:100%; max-width:520px; box-shadow:var(--shadow-lg); animation:slideUp 0.3s cubic-bezier(0.34,1.56,0.64,1); overflow:hidden; }",
    "@keyframes slideUp { from{opacity:0;transform:translateY(30px) scale(0.96);} to{opacity:1;transform:translateY(0) scale(1);} }",
    ".modal-header { padding:24px 28px 20px; border-bottom:1px solid var(--border-subtle); }",
    ".modal-brand { display:flex; align-items:center; gap:14px; margin-bottom:4px; }",
    ".modal-brand-icon { width:46px; height:46px; border-radius:14px; background:linear-gradient(135deg,#6366f1,#8b5cf6); display:flex; align-items:center; justify-content:center; font-size:22px; box-shadow:0 8px 24px rgba(99,102,241,0.4); }",
    ".modal-header h2 { font-size:20px; font-weight:700; color:var(--text-primary); letter-spacing:-0.3px; }",
    ".modal-header p { font-size:13px; color:var(--text-secondary); margin-top:4px; line-height:1.5; }",
    ".modal-body { padding:24px 28px; display:flex; flex-direction:column; gap:18px; }",
    ".form-group { display:flex; flex-direction:column; gap:6px; }",
    ".form-label { font-size:12px; font-weight:600; color:var(--text-secondary); text-transform:uppercase; letter-spacing:0.6px; display:flex; align-items:center; gap:6px; }",
    ".form-label span.required { color:#f87171; }",
    ".form-input { padding:11px 14px; border-radius:var(--radius-md); background:var(--bg-input); border:1px solid var(--border-medium); color:var(--text-primary); font-family:inherit; font-size:13px; outline:none; transition:var(--transition); width:100%; }",
    ".form-input:focus { border-color:var(--border-accent); box-shadow:0 0 0 3px var(--accent-glow); }",
    ".form-input::placeholder { color:var(--text-muted); }",
    ".form-hint { font-size:11px; color:var(--text-muted); line-height:1.4; }",
    ".modal-footer { padding:16px 28px 24px; display:flex; flex-direction:column; gap:10px; }",
    ".btn-primary { width:100%; padding:13px; border-radius:var(--radius-md); background:linear-gradient(135deg,#6366f1,#8b5cf6); border:none; color:white; font-family:inherit; font-size:14px; font-weight:600; cursor:pointer; transition:var(--transition); box-shadow:0 4px 20px rgba(99,102,241,0.4); }",
    ".btn-primary:hover:not(:disabled) { transform:translateY(-1px); box-shadow:0 8px 30px rgba(99,102,241,0.5); }",
    ".btn-primary:disabled { opacity:0.5; cursor:not-allowed; }",
    ".btn-secondary { width:100%; padding:11px; border-radius:var(--radius-md); background:transparent; border:1px solid var(--border-medium); color:var(--text-secondary); font-family:inherit; font-size:13px; font-weight:500; cursor:pointer; transition:var(--transition); }",
    ".btn-secondary:hover { background:var(--bg-glass); border-color:var(--border-accent); color:var(--text-primary); }",
    ".error-banner { padding:10px 14px; border-radius:var(--radius-md); background:rgba(239,68,68,0.1); border:1px solid rgba(239,68,68,0.25); color:#fca5a5; font-size:13px; display:flex; align-items:flex-start; gap:8px; line-height:1.45; }",
    ".error-banner svg { width:16px; height:16px; flex-shrink:0; margin-top:1px; color:#f87171; }",
    ".status-chip { display:inline-flex; align-items:center; gap:5px; padding:4px 10px; border-radius:var(--radius-full); font-size:11px; font-weight:500; }",
    ".status-chip.connected { background:rgba(34,197,94,0.12); color:#86efac; border:1px solid rgba(34,197,94,0.2); }",
    ".status-chip.disconnected { background:rgba(245,158,11,0.12); color:#fcd34d; border:1px solid rgba(245,158,11,0.2); }",
    ".status-chip-dot { width:6px; height:6px; border-radius:50%; }",
    ".status-chip.connected .status-chip-dot { background:#22c55e; box-shadow:0 0 6px rgba(34,197,94,0.6); animation:pulse-green 2s infinite; }",
    ".status-chip.disconnected .status-chip-dot { background:#f59e0b; }",
    "@keyframes pulse-green { 0%,100%{box-shadow:0 0 4px rgba(34,197,94,0.5);}50%{box-shadow:0 0 10px rgba(34,197,94,0.9);} }",
    "* { scrollbar-width:thin; scrollbar-color:var(--border-medium) transparent; }",
    "@media (max-width:768px) { .sidebar{display:none;} .messages-container{padding:0 12px;} .topbar{padding:12px 16px;} .input-area{padding:12px 12px 16px;} .message-body{max-width:88%;} }",
    "",
])
(BASE / "src" / "index.css").write_text(css)
print("src/index.css written.")


In [ ]:
# Step 6.9 — src/App.jsx
app_jsx = "\n".join([
    "import { useState, useEffect, useCallback } from 'react';",
    "import ConfigModal from './components/ConfigModal';",
    "import Sidebar from './components/Sidebar';",
    "import TopBar from './components/TopBar';",
    "import ChatMessages from './components/ChatMessages';",
    "import ChatInput from './components/ChatInput';",
    "import './index.css';",
    "",
    "const STORAGE_KEY = 'support_bot_config';",
    "const SESSIONS_KEY = 'support_bot_sessions';",
    "const API_VERSION = '2024-12-01-preview';",
    "",
    "function App() {",
    "  const [config, setConfig] = useState(null);",
    "  const [showConfigModal, setShowConfigModal] = useState(false);",
    "  const [sessions, setSessions] = useState([]);",
    "  const [activeSessionId, setActiveSessionId] = useState(null);",
    "  const [isStreaming, setIsStreaming] = useState(false);",
    "",
    "  useEffect(() => {",
    "    const saved = localStorage.getItem(STORAGE_KEY);",
    "    if (saved) { try { setConfig(JSON.parse(saved)); } catch {} } else { setShowConfigModal(true); }",
    "    const savedSessions = localStorage.getItem(SESSIONS_KEY);",
    "    if (savedSessions) { try { const s=JSON.parse(savedSessions); setSessions(s); if(s.length>0) setActiveSessionId(s[0].id); } catch {} }",
    "  }, []);",
    "",
    "  const saveConfig = (cfg) => {",
    "    localStorage.setItem(STORAGE_KEY, JSON.stringify(cfg));",
    "    setConfig(cfg); setShowConfigModal(false);",
    "    if (sessions.length === 0) createNewSession(cfg);",
    "  };",
    "",
    "  const createNewSession = useCallback((cfg = config) => {",
    "    if (!cfg) { setShowConfigModal(true); return; }",
    "    const id = Date.now().toString();",
    "    const newSession = { id, title: 'New conversation', messages: [], createdAt: new Date().toISOString() };",
    "    setSessions(prev => { const updated=[newSession,...prev]; localStorage.setItem(SESSIONS_KEY,JSON.stringify(updated)); return updated; });",
    "    setActiveSessionId(id);",
    "  }, [config, sessions]);",
    "",
    "  const activeSession = sessions.find(s => s.id === activeSessionId);",
    "",
    "  const updateSession = useCallback((id, messages) => {",
    "    setSessions(prev => {",
    "      const updated = prev.map(s => {",
    "        if (s.id !== id) return s;",
    "        let title = s.title;",
    "        if (title === 'New conversation' && messages.length > 0) {",
    "          const firstUser = messages.find(m => m.role === 'user');",
    "          if (firstUser) title = firstUser.content.slice(0,40) + (firstUser.content.length>40?'...':'');",
    "        }",
    "        return { ...s, messages, title };",
    "      });",
    "      localStorage.setItem(SESSIONS_KEY, JSON.stringify(updated));",
    "      return updated;",
    "    });",
    "  }, []);",
    "",
    "  const sendMessage = useCallback(async (userText) => {",
    "    if (!config || !activeSession || isStreaming) return;",
    "    const userMsg = { role:'user', content:userText, timestamp:new Date().toISOString() };",
    "    const newMessages = [...activeSession.messages, userMsg];",
    "    updateSession(activeSession.id, newMessages);",
    "    setIsStreaming(true);",
    "    const botMsgId = `bot-${Date.now()}`;",
    "    const botMsg = { role:'assistant', content:'', timestamp:new Date().toISOString(), id:botMsgId, streaming:true };",
    "    updateSession(activeSession.id, [...newMessages, botMsg]);",
    "    const endpoint = config.endpoint.replace(/\/$/,'');",
    "    const url = `${endpoint}/openai/deployments/${config.deployment}/chat/completions?api-version=${API_VERSION}`;",
    "    const allMessages = [{ role:'system', content:config.systemPrompt||'You are a helpful support assistant.' }, ...newMessages.map(({role,content})=>({role,content}))];",
    "    try {",
    "      const res = await fetch(url, { method:'POST', headers:{'Content-Type':'application/json','api-key':config.apiKey}, body:JSON.stringify({messages:allMessages,stream:true,max_tokens:4096,temperature:0.7,top_p:1.0}) });",
    "      if (!res.ok) { const e=await res.json().catch(()=>({})); throw new Error(e?.error?.message||`HTTP ${res.status}`); }",
    "      const reader=res.body.getReader(); const decoder=new TextDecoder(); let fullContent=''; let buffer='';",
    "      while (true) {",
    "        const {done,value}=await reader.read(); if(done) break;",
    "        buffer+=decoder.decode(value,{stream:true});",
    "        const lines=buffer.split('\\n'); buffer=lines.pop();",
    "        for (const line of lines) {",
    "          if (!line.startsWith('data: ')) continue;",
    "          const data=line.slice(6).trim(); if(data==='[DONE]') break;",
    "          try { const parsed=JSON.parse(data); const delta=parsed?.choices?.[0]?.delta?.content;",
    "            if (delta) { fullContent+=delta; setSessions(prev=>{ const updated=prev.map(s=>{ if(s.id!==activeSession.id) return s; return {...s,messages:s.messages.map(m=>m.id===botMsgId?{...m,content:fullContent}:m)}; }); localStorage.setItem(SESSIONS_KEY,JSON.stringify(updated)); return updated; }); }",
    "          } catch {}",
    "        }",
    "      }",
    "      setSessions(prev=>{ const updated=prev.map(s=>{ if(s.id!==activeSession.id) return s; return {...s,messages:s.messages.map(m=>m.id===botMsgId?{...m,streaming:false}:m)}; }); localStorage.setItem(SESSIONS_KEY,JSON.stringify(updated)); return updated; });",
    "    } catch(err) {",
    "      setSessions(prev=>{ const updated=prev.map(s=>{ if(s.id!==activeSession.id) return s; return {...s,messages:s.messages.map(m=>m.id===botMsgId?{...m,content:`Error: ${err.message}`,streaming:false,error:true}:m)}; }); localStorage.setItem(SESSIONS_KEY,JSON.stringify(updated)); return updated; });",
    "    } finally { setIsStreaming(false); }",
    "  }, [config, activeSession, isStreaming, updateSession]);",
    "",
    "  const clearSession = useCallback(() => { if (!activeSession) return; updateSession(activeSession.id, []); }, [activeSession, updateSession]);",
    "",
    "  return (",
    "    <div className=\"app\">",
    "      {(showConfigModal || !config) && (<ConfigModal onSave={saveConfig} existingConfig={config} onClose={config ? () => setShowConfigModal(false) : null} />)}",
    "      <Sidebar sessions={sessions} activeSessionId={activeSessionId} onSelectSession={setActiveSessionId} onNewChat={createNewSession} onOpenSettings={() => setShowConfigModal(true)} />",
    "      <main className=\"main-content\">",
    "        <TopBar botName={config?.botName || 'Support Assistant'} isConnected={!!config && !showConfigModal} onOpenSettings={() => setShowConfigModal(true)} onClear={clearSession} />",
    "        <ChatMessages messages={activeSession?.messages || []} isStreaming={isStreaming} onChipClick={sendMessage} />",
    "        <ChatInput onSend={sendMessage} disabled={!config || isStreaming} isStreaming={isStreaming} placeholder={!config ? 'Configure API key to start...' : 'Type your message...'} />",
    "      </main>",
    "    </div>",
    "  );",
    "}",
    "",
    "export default App;",
    "",
])
(BASE / "src" / "App.jsx").write_text(app_jsx)
print("src/App.jsx written.")


In [ ]:
# Step 6.10 — src/components/ChatInput.jsx
chat_input = "\n".join([
    "import { useState, useRef, useEffect } from 'react';",
    "",
    "export default function ChatInput({ onSend, disabled, isStreaming, placeholder }) {",
    "  const [text, setText] = useState('');",
    "  const textareaRef = useRef(null);",
    "  useEffect(() => {",
    "    const ta = textareaRef.current; if (!ta) return;",
    "    ta.style.height = 'auto';",
    "    ta.style.height = Math.min(ta.scrollHeight, 140) + 'px';",
    "  }, [text]);",
    "  const handleSend = () => {",
    "    const msg = text.trim(); if (!msg || disabled) return;",
    "    setText(''); onSend(msg);",
    "  };",
    "  const handleKeyDown = (e) => {",
    "    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); handleSend(); }",
    "  };",
    "  return (",
    "    <div className=\"input-area\">",
    "      <div className=\"input-container\">",
    "        <div className=\"input-box\">",
    "          <textarea ref={textareaRef} rows={1} value={text} onChange={e => setText(e.target.value)} onKeyDown={handleKeyDown} placeholder={placeholder || 'Type your message...'} disabled={disabled && !isStreaming} />",
    "          <button className=\"send-btn\" onClick={handleSend} disabled={!text.trim() || (disabled && !isStreaming)} title=\"Send (Enter)\">",
    "            {isStreaming ? (<svg viewBox=\"0 0 24 24\" fill=\"currentColor\"><rect x=\"6\" y=\"6\" width=\"12\" height=\"12\" rx=\"2\" /></svg>)",
    "              : (<svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2.5\"><line x1=\"22\" y1=\"2\" x2=\"11\" y2=\"13\" /><polygon points=\"22 2 15 22 11 13 2 9 22 2\" /></svg>)}",
    "          </button>",
    "        </div>",
    "        <p className=\"input-footer\">Press Enter to send · Shift+Enter for new line · Powered by Azure GPT-4o</p>",
    "      </div>",
    "    </div>",
    "  );",
    "}",
    "",
])
(BASE / "src" / "components" / "ChatInput.jsx").write_text(chat_input)
print("src/components/ChatInput.jsx written.")


In [ ]:
# Step 6.11 — src/components/ChatMessages.jsx
chat_messages = "\n".join([
    "import { useEffect, useRef } from 'react';",
    "",
    "const SUGGESTIONS = ['How can I track my order?','What is your return policy?','I need help with my account','Talk to a human agent'];",
    "",
    "function formatTime(iso) { if(!iso) return ''; return new Date(iso).toLocaleTimeString([],{hour:'2-digit',minute:'2-digit'}); }",
    "",
    "function MessageBubble({ message }) {",
    "  const isUser = message.role === 'user';",
    "  return (",
    "    <div className={`message-group ${isUser?'user':'bot'}`}>",
    "      <div className={`message-avatar ${isUser?'user-avatar-sm':'bot-avatar-sm'}`}>{isUser?'\ud83d\udc64':'\ud83e\udd16'}</div>",
    "      <div className=\"message-body\">",
    "        <div className={`message-bubble ${isUser?'user':'bot'} ${message.error?'error-msg':''}`} style={message.error?{borderColor:'rgba(239,68,68,0.3)',background:'rgba(239,68,68,0.08)'}:{}}>",
    "          {message.content}",
    "          {message.streaming && message.content.length>0 && (<span style={{display:'inline-block',width:'2px',height:'14px',background:'var(--text-accent)',marginLeft:'2px',verticalAlign:'middle',animation:'cursorBlink 0.8s steps(1) infinite'}} />)}",
    "        </div>",
    "        <span className=\"message-time\">{formatTime(message.timestamp)}</span>",
    "      </div>",
    "    </div>",
    "  );",
    "}",
    "",
    "function TypingIndicator() {",
    "  return (<div className=\"message-group bot\"><div className=\"message-avatar bot-avatar-sm\">\ud83e\udd16</div><div className=\"message-body\"><div className=\"typing-bubble\"><div className=\"typing-dot\" /><div className=\"typing-dot\" /><div className=\"typing-dot\" /></div></div></div>);",
    "}",
    "",
    "export default function ChatMessages({ messages, isStreaming, onChipClick }) {",
    "  const bottomRef = useRef(null);",
    "  useEffect(() => { bottomRef.current?.scrollIntoView({behavior:'smooth'}); }, [messages, isStreaming]);",
    "  const isEmpty = messages.length === 0;",
    "  const lastIsBot = messages.length>0 && messages[messages.length-1].role==='assistant';",
    "  const showTyping = isStreaming && (!lastIsBot || (lastIsBot && messages[messages.length-1].content===''));",
    "  return (",
    "    <div className=\"messages-wrapper\">",
    "      <style>{`@keyframes cursorBlink { 0%,100%{opacity:1;} 50%{opacity:0;} }`}</style>",
    "      <div className=\"messages-container\">",
    "        {isEmpty ? (",
    "          <div className=\"welcome-screen\">",
    "            <div className=\"welcome-icon\">\ud83e\udd16</div>",
    "            <h2>How can I help you today?</h2>",
    "            <p>I'm your AI-powered support assistant. Ask me anything about products, orders, accounts, or general support.</p>",
    "            <div className=\"welcome-chips\">{SUGGESTIONS.map(s=>(<button key={s} className=\"welcome-chip\" onClick={()=>onChipClick(s)}>{s}</button>))}</div>",
    "          </div>",
    "        ) : (<>{messages.map((msg,i)=>(<MessageBubble key={msg.id||i} message={msg} />))}{showTyping && <TypingIndicator />}</>)}",
    "        <div ref={bottomRef} />",
    "      </div>",
    "    </div>",
    "  );",
    "}",
    "",
])
(BASE / "src" / "components" / "ChatMessages.jsx").write_text(chat_messages)
print("src/components/ChatMessages.jsx written.")


In [ ]:
# Step 6.12 — src/components/ConfigModal.jsx
config_modal = "\n".join([
    "import { useState } from 'react';",
    "",
    "const EyeIcon = ({show}) => show",
    "  ? (<svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><path d=\"M1 12s4-8 11-8 11 8 11 8-4 8-11 8-11-8-11-8z\" /><circle cx=\"12\" cy=\"12\" r=\"3\" /></svg>)",
    "  : (<svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><path d=\"M17.94 17.94A10.07 10.07 0 0 1 12 20c-7 0-11-8-11-8a18.45 18.45 0 0 1 5.06-5.94M9.9 4.24A9.12 9.12 0 0 1 12 4c7 0 11 8 11 8a18.5 18.5 0 0 1-2.16 3.19m-6.72-1.07a3 3 0 1 1-4.24-4.24\" /><line x1=\"1\" y1=\"1\" x2=\"23\" y2=\"23\" /></svg>);",
    "",
    "export default function ConfigModal({ onSave, existingConfig, onClose }) {",
    "  const [form, setForm] = useState({",
    "    endpoint:     existingConfig?.endpoint     || 'https://your-resource.cognitiveservices.azure.com/',",
    "    deployment:   existingConfig?.deployment   || 'gpt-4o-2024-08-06-project-demo',",
    "    apiKey:       existingConfig?.apiKey       || '',",
    "    botName:      existingConfig?.botName      || 'Support Assistant',",
    "    systemPrompt: existingConfig?.systemPrompt || 'You are a helpful and professional customer support assistant. Be concise, friendly, and empathetic.',",
    "  });",
    "  const [showKey, setShowKey] = useState(false);",
    "  const [error, setError]     = useState('');",
    "  const [testing, setTesting] = useState(false);",
    "  const handleChange = (e) => { setForm(f=>({...f,[e.target.name]:e.target.value})); setError(''); };",
    "  const handleSave = () => {",
    "    if (!form.apiKey.trim())    { setError('API Key is required.'); return; }",
    "    if (!form.endpoint.trim())  { setError('Endpoint URL is required.'); return; }",
    "    if (!form.deployment.trim()){ setError('Deployment name is required.'); return; }",
    "    onSave(form);",
    "  };",
    "  const handleTest = async () => {",
    "    if (!form.apiKey.trim()||!form.endpoint.trim()||!form.deployment.trim()) { setError('Fill all fields first.'); return; }",
    "    setTesting(true); setError('');",
    "    try {",
    "      const ep = form.endpoint.replace(/\/$/,'');",
    "      const url = `${ep}/openai/deployments/${form.deployment}/chat/completions?api-version=2024-12-01-preview`;",
    "      const res = await fetch(url, { method:'POST', headers:{'Content-Type':'application/json','api-key':form.apiKey},",
    "        body:JSON.stringify({messages:[{role:'system',content:'Reply: Connection successful!'},{role:'user',content:'Hello'}],max_tokens:20,stream:false}) });",
    "      if (res.ok) setError('\u2705 Connection successful!');",
    "      else { const d=await res.json().catch(()=>({})); setError(`\u274c ${d?.error?.message||'HTTP '+res.status}`); }",
    "    } catch(e) { setError(`\u274c Network error: ${e.message}`); }",
    "    finally { setTesting(false); }",
    "  };",
    "  const isValid = form.apiKey.trim() && form.endpoint.trim() && form.deployment.trim();",
    "  return (",
    "    <div className=\"modal-overlay\"><div className=\"modal\">",
    "      <div className=\"modal-header\"><div className=\"modal-brand\"><div className=\"modal-brand-icon\">\ud83e\udd16</div><div><h2>Configure Your Bot</h2><p>Connect your Azure OpenAI deployment to get started</p></div></div></div>",
    "      <div className=\"modal-body\">",
    "        {error && (<div className=\"error-banner\"><svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><circle cx=\"12\" cy=\"12\" r=\"10\" /><line x1=\"12\" y1=\"8\" x2=\"12\" y2=\"12\" /><line x1=\"12\" y1=\"16\" x2=\"12.01\" y2=\"16\" /></svg><span style={{color:error.startsWith('\u2705')?'#86efac':undefined}}>{error}</span></div>)}",
    "        <div className=\"form-group\"><label className=\"form-label\">API Key <span className=\"required\">*</span></label><div style={{position:'relative'}}><input name=\"apiKey\" type={showKey?'text':'password'} className=\"form-input\" placeholder=\"Enter your Azure OpenAI API key\" value={form.apiKey} onChange={handleChange} style={{paddingRight:'40px'}} /><button onClick={()=>setShowKey(s=>!s)} style={{position:'absolute',right:'10px',top:'50%',transform:'translateY(-50%)',background:'none',border:'none',color:'var(--text-muted)',cursor:'pointer',display:'flex',padding:'4px',width:'20px',height:'20px'}}><EyeIcon show={showKey} /></button></div><span className=\"form-hint\">Stored locally, only sent to Azure.</span></div>",
    "        <div className=\"form-group\"><label className=\"form-label\">Azure Endpoint <span className=\"required\">*</span></label><input name=\"endpoint\" type=\"text\" className=\"form-input\" placeholder=\"https://your-resource.cognitiveservices.azure.com/\" value={form.endpoint} onChange={handleChange} /></div>",
    "        <div className=\"form-group\"><label className=\"form-label\">Deployment Name <span className=\"required\">*</span></label><input name=\"deployment\" type=\"text\" className=\"form-input\" placeholder=\"gpt-4o-2024-08-06-project-demo\" value={form.deployment} onChange={handleChange} /></div>",
    "        <div className=\"form-group\"><label className=\"form-label\">Bot Name</label><input name=\"botName\" type=\"text\" className=\"form-input\" placeholder=\"Support Assistant\" value={form.botName} onChange={handleChange} /></div>",
    "        <div className=\"form-group\"><label className=\"form-label\">System Prompt</label><textarea name=\"systemPrompt\" className=\"form-input\" value={form.systemPrompt} onChange={handleChange} rows={3} style={{resize:'vertical',minHeight:'72px'}} /><span className=\"form-hint\">Defines bot behaviour.</span></div>",
    "      </div>",
    "      <div className=\"modal-footer\"><button className=\"btn-primary\" onClick={handleSave} disabled={!isValid}>Save &amp; Start Chatting</button><button className=\"btn-secondary\" onClick={handleTest} disabled={!isValid||testing}>{testing?'Testing...':'Test Connection'}</button>{onClose&&(<button className=\"btn-secondary\" onClick={onClose}>Cancel</button>)}</div>",
    "    </div></div>",
    "  );",
    "}",
    "",
])
(BASE / "src" / "components" / "ConfigModal.jsx").write_text(config_modal)
print("src/components/ConfigModal.jsx written.")


In [ ]:
# Step 6.13 — src/components/Sidebar.jsx
sidebar = "\n".join([
    "export default function Sidebar({ sessions, activeSessionId, onSelectSession, onNewChat, onOpenSettings }) {",
    "  return (",
    "    <aside className=\"sidebar\">",
    "      <div className=\"sidebar-header\">",
    "        <div className=\"brand\">",
    "          <div className=\"brand-icon\">\ud83e\udd16</div>",
    "          <div className=\"brand-text\"><h1>SupportBot AI</h1><p>Powered by Azure GPT-4o</p></div>",
    "        </div>",
    "        <button className=\"new-chat-btn\" onClick={onNewChat}>",
    "          <svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><line x1=\"12\" y1=\"5\" x2=\"12\" y2=\"19\" /><line x1=\"5\" y1=\"12\" x2=\"19\" y2=\"12\" /></svg>",
    "          New Conversation",
    "        </button>",
    "      </div>",
    "      <div className=\"sidebar-content\">",
    "        {sessions.length > 0 && (<>",
    "          <div className=\"sidebar-section-label\">Recent</div>",
    "          {sessions.map(s => (<div key={s.id} className={`conversation-item ${s.id===activeSessionId?'active':''}`} onClick={()=>onSelectSession(s.id)}><span className=\"conversation-item-icon\">\ud83d\udcac</span><span className=\"conversation-item-text\">{s.title}</span></div>))}",
    "        </>)}",
    "        {sessions.length === 0 && (<div style={{padding:'20px 8px',color:'var(--text-muted)',fontSize:'13px',lineHeight:1.5}}>No conversations yet. Start a new chat!</div>)}",
    "      </div>",
    "      <div className=\"sidebar-footer\">",
    "        <button className=\"settings-btn\" onClick={onOpenSettings}>",
    "          <svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><circle cx=\"12\" cy=\"12\" r=\"3\" /><path d=\"M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1-2.83 2.83l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-4 0v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83-2.83l.06-.06A1.65 1.65 0 0 0 4.68 15a1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1 0-4h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 2.83-2.83l.06.06A1.65 1.65 0 0 0 9 4.68a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 4 0v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 2.83l-.06.06A1.65 1.65 0 0 0 19.4 9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 0 4h-.09a1.65 1.65 0 0 0-1.51 1z\" /></svg>",
    "          API Settings",
    "        </button>",
    "      </div>",
    "    </aside>",
    "  );",
    "}",
    "",
])
(BASE / "src" / "components" / "Sidebar.jsx").write_text(sidebar)
print("src/components/Sidebar.jsx written.")


In [ ]:
# Step 6.14 — src/components/TopBar.jsx
topbar = "\n".join([
    "export default function TopBar({ botName, isConnected, onOpenSettings, onClear }) {",
    "  return (",
    "    <div className=\"topbar\">",
    "      <div className=\"topbar-bot-info\">",
    "        <div className=\"bot-avatar\">\ud83e\udd16<div className={`status-dot ${isConnected?'':'offline'}`} /></div>",
    "        <div><h2>{botName}</h2><p className={isConnected?'':'offline-status'}>{isConnected?'Online \u00b7 Ready to help':'Configure API key to start'}</p></div>",
    "      </div>",
    "      <div className=\"topbar-actions\">",
    "        <div className={`status-chip ${isConnected?'connected':'disconnected'}`}><div className=\"status-chip-dot\" />{isConnected?'Connected':'Disconnected'}</div>",
    "        <button className=\"icon-btn\" onClick={onClear} title=\"Clear conversation\"><svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><polyline points=\"3 6 5 6 21 6\" /><path d=\"M19 6l-1 14a2 2 0 0 1-2 2H8a2 2 0 0 1-2-2L5 6\" /><path d=\"M10 11v6\" /><path d=\"M14 11v6\" /></svg></button>",
    "        <button className=\"icon-btn\" onClick={onOpenSettings} title=\"API Settings\"><svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" strokeWidth=\"2\"><circle cx=\"12\" cy=\"12\" r=\"3\" /><path d=\"M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1-2.83 2.83l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-4 0v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83-2.83l.06-.06A1.65 1.65 0 0 0 4.68 15a1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1 0-4h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 2.83-2.83l.06.06A1.65 1.65 0 0 0 9 4.68a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 4 0v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 2.83l-.06.06A1.65 1.65 0 0 0 19.4 9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 0 4h-.09a1.65 1.65 0 0 0-1.51 1z\" /></svg></button>",
    "      </div>",
    "    </div>",
    "  );",
    "}",
    "",
])
(BASE / "src" / "components" / "TopBar.jsx").write_text(topbar)
print("src/components/TopBar.jsx written.")


In [ ]:
# Step 6.15 — server/index.js  (Express + Azure OpenAI SSE streaming)
server_lines = [
    "import express from 'express';",
    "import cors from 'cors';",
    "import { AzureOpenAI } from 'openai';",
    "",
    "const app = express();",
    "app.use(cors()); app.use(express.json());",
    "",
    "app.get('/health', (req, res) => res.json({ status: 'ok' }));",
    "",
    "app.post('/api/chat', async (req, res) => {",
    "  const { messages, apiKey, endpoint, deployment, systemPrompt } = req.body;",
    "  if (!apiKey || !endpoint || !deployment) return res.status(400).json({ error: 'Missing apiKey, endpoint, or deployment' });",
    "  try {",
    "    const client = new AzureOpenAI({ apiVersion: '2024-12-01-preview', endpoint, apiKey });",
    "    const allMessages = [{ role: 'system', content: systemPrompt || 'You are a helpful customer support assistant.' }, ...messages];",
    "    res.setHeader('Content-Type', 'text/event-stream');",
    "    res.setHeader('Cache-Control', 'no-cache');",
    "    res.setHeader('Connection', 'keep-alive');",
    "    res.setHeader('X-Accel-Buffering', 'no');",
    "    res.flushHeaders();",
    "    const stream = await client.chat.completions.create({ stream: true, messages: allMessages, max_tokens: 4096, temperature: 0.7, top_p: 1.0, model: deployment });",
    "    for await (const chunk of stream) {",
    "      if (chunk.choices && chunk.choices[0]) {",
    "        const delta = chunk.choices[0].delta;",
    "        if (delta && delta.content) res.write(`data: ${JSON.stringify({ content: delta.content })}\\n\\n`);",
    "        if (chunk.choices[0].finish_reason === 'stop') res.write('data: [DONE]\\n\\n');",
    "      }",
    "    }",
    "    res.end();",
    "  } catch (err) {",
    "    const msg = err?.message || 'Unknown error';",
    "    if (!res.headersSent) res.status(500).json({ error: msg });",
    "    else { res.write(`data: ${JSON.stringify({ error: msg })}\\n\\n`); res.end(); }",
    "  }",
    "});",
    "",
    "const PORT = process.env.PORT || 3001;",
    "app.listen(PORT, () => console.log(`Server running on http://localhost:${PORT}`));",
    "",
]
(BASE / "server" / "index.js").write_text("\n".join(server_lines))

import json as _j
srv_pkg = {"name":"support-bot-server","version":"1.0.0","type":"module","main":"index.js","scripts":{"start":"node index.js"},"dependencies":{"cors":"^2.8.5","express":"^4.21.2","openai":"^4.103.0"}}
(BASE / "server" / "package.json").write_text(_j.dumps(srv_pkg, indent=2))

(BASE / "public" / "vite.svg").write_text('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100"><text y=".9em" font-size="90">\u26a1</text></svg>\n')

print("server/index.js written.")
print("server/package.json written.")
print("public/vite.svg written.")


In [ ]:
# Step 6.16 — Verify complete file tree
import os
print("Complete project tree:")
for root, dirs, files in os.walk(BASE):
    dirs[:] = sorted(d for d in dirs if d != "node_modules")
    depth = root.replace(str(BASE), '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}📁 {os.path.basename(root) or BASE}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f"  {indent}📄 {f}  ({size:,} B)")


---
## 🧪 Step 7 — LangChain 1.2.0 Inference Test


In [ ]:
# Step 7.1 — Single-turn support chain (LangChain 1.2.0 LCEL)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

support_chain = (
    ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("human", "{msg}")])
    | llm_fast
    | StrOutputParser()
)

query = "I ordered a product 5 days ago but haven't received a shipping confirmation."
print(f"User: {query}\n")
print("Bot:", support_chain.invoke({"msg": query}))


In [ ]:
# Step 7.2 — Multi-turn conversation (LangChain 1.2.0 MessagesPlaceholder)
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

multi_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{msg}"),
])
multi_chain = multi_prompt | llm_fast | StrOutputParser()

history = []
turns = ["My order hasn't arrived yet.", "Order number is ORD-2024-98765.", "Can I get expedited shipping as compensation?"]

print("=" * 55)
for t in turns:
    print(f"User: {t}")
    resp = multi_chain.invoke({"history": history, "msg": t})
    print(f"Bot:  {resp}\n")
    history += [HumanMessage(content=t), AIMessage(content=resp)]
print("Multi-turn test passed.")


---
## 📋 Step 8 — Review Dataset Samples & Charts


In [ ]:
# Step 8.1 — Show 5 random training samples
import jsonlines, random

with jsonlines.open(TRAIN_PATH) as r:
    all_samples = list(r)

for idx in random.sample(range(len(all_samples)), min(5, len(all_samples))):
    s = all_samples[idx]
    print(f"--- Example #{idx+1} ---")
    for m in s["messages"]:
        snippet = m["content"][:180] + ("..." if len(m["content"])>180 else "")
        print(f"  [{m['role'].upper()}]: {snippet}")
    print()


In [ ]:
# Step 8.2 — Category distribution bar chart
import matplotlib.pyplot as plt, jsonlines

kw_map = {"order":"order_tracking","track":"order_tracking","return":"returns_refunds","refund":"returns_refunds",
          "account":"account_issues","password":"account_issues","warranty":"product_questions","product":"product_questions",
          "app":"technical_support","error":"technical_support","manager":"escalation","project":"ai_project_mentor","ml":"ai_project_mentor"}

cats = {c: 0 for c in TOPIC_CATEGORIES}
with jsonlines.open(TRAIN_PATH) as r:
    for obj in r:
        text = " ".join(m["content"].lower() for m in obj["messages"] if m["role"]=="user")
        for kw, cat in kw_map.items():
            if kw in text and cat in cats:
                cats[cat] += 1; break

labels = [c.replace("_"," ").title() for c in cats]
values = list(cats.values())
colors = ["#6366f1","#8b5cf6","#a855f7","#06b6d4","#14b8a6","#22c55e","#f59e0b"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, values, color=colors[:len(labels)], edgecolor="white")
ax.set_xlabel("Examples"); ax.set_title("Training Dataset Category Distribution", fontweight="bold")
for b, v in zip(bars, values):
    ax.text(b.get_width()+1, b.get_y()+b.get_height()/2, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.savefig(BASE / "DATA JSONL FILES" / "category_distribution.png", dpi=120)
plt.show()


---

# ═══════════════════════════════════════════════════
# 🏗️  PART II — DEPLOYMENT MANUAL
# ═══════════════════════════════════════════════════

> The cells below are **instructional** — read and follow in the Azure / AWS consoles.
> No code execution is needed unless explicitly marked.

---

## Phase 1 · Azure AI Foundry — Fine-Tune GPT-4o

### Prerequisites

| Requirement | Detail |
|-------------|--------|
| Azure Subscription | Active with OpenAI access approved |
| Files ready | `train.jsonl` & `validation.jsonl` from Step 4 |
| Quota | GPT-4o fine-tuning quota in your region |

---

### ➡️ A1 — Upload Data to Azure AI Foundry

1. Open **[ai.azure.com](https://ai.azure.com)** → sign in → select your Hub/Project.
2. Left sidebar → **Fine-tuning** → **+ Fine-tune a model**.
3. Base model: **GPT-4o** (`gpt-4o-2024-08-06`).
4. Upload **Training data** → `DATA JSONL FILES/train.jsonl`.
5. Upload **Validation data** → `DATA JSONL FILES/validation.jsonl`.
6. Azure auto-validates — both must show ✅ before proceeding.

---

### ➡️ A2 — Hyperparameter Configuration

| Parameter | Recommended | Notes |
|-----------|-------------|-------|
| Epochs | `3` | Increase to 5 for small datasets |
| Batch size | `auto` | Azure-managed |
| Learning rate multiplier | `1.0` | Raise to 2.0 if loss plateaus |
| Suffix | `project-demo` | Appears in deployment name |

> 💰 **Cost estimate:** 400 examples × ~150 tokens × 3 epochs ≈ 180 K tokens ≈ **$4.50**

---

### ➡️ A3 — Submit & Monitor Job

1. Click **Submit** → job appears under **Fine-tuning → Jobs**.
2. States: `Queued → Running → Succeeded ✅`
3. Typical duration: **30–90 min** for 400 examples.
4. Note the **fine-tuned model name** on completion.

---

### ➡️ A4 — Deploy the Fine-Tuned Model

1. Click **Deploy** on the succeeded model.
2. Deployment name: `gpt-4o-2024-08-06-project-demo`
3. TPM (tokens-per-minute): set to your quota (e.g. 10,000).
4. Status → **Succeeded ✅**
5. Copy from **Keys and Endpoint**:
   - `AZURE_ENDPOINT` = `https://<resource>.cognitiveservices.azure.com/`
   - `AZURE_API_KEY`  = Key 1
   - `AZURE_DEPLOYMENT` = `gpt-4o-2024-08-06-project-demo`

---

### ➡️ A5 — Test in Playground

1. **Playground → Chat** → select your fine-tuned deployment.
2. System prompt: paste `SYSTEM_PROMPT` from Step 3.
3. Send a few test messages → verify mentor/support tone. ✅

---

## Phase 2 · Local React App Setup

### ➡️ B1 — Install & Run

```bash
cd project-root/
npm install          # install React deps
npm run dev          # http://localhost:5173
```

In the Config Modal enter `AZURE_ENDPOINT`, `AZURE_API_KEY`, `AZURE_DEPLOYMENT`.
Click **Test Connection ✅** → **Save & Start Chatting**.

### ➡️ B2 — Production Build

```bash
npm run build        # outputs to ./dist/
npm run preview      # preview locally
```

### ➡️ B3 — Push to GitHub

```bash
git init
git add .
git commit -m "feat: Azure GPT-4o fine-tuned support chatbot"
git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPO.git
git push -u origin main
```

> ⚠️ Ensure `node_modules/` and `dist/` are in `.gitignore` (already done in Step 6.4).

---

## Phase 3 · AWS CI/CD — GitHub → CodePipeline → S3

### ➡️ C1 — Create S3 Bucket

1. AWS Console → **S3** → **Create bucket**.
2. Name: `my-react-cicd-demo` *(globally unique)*
3. Region: your choice (e.g. `us-east-1`).
4. Leave **Block Public Access** ON for now.
5. **Create bucket** → leave empty.

---

### ➡️ C2 — Verify `buildspec.yml` at Repo Root

The file was written in Step 6.3. Confirm it exists at:
`project-root/buildspec.yml`

```yaml
version: 0.2
phases:
  install:
    runtime-versions:
      nodejs: 18
    commands:
      - npm ci --legacy-peer-deps
  build:
    commands:
      - npm run build
artifacts:
  files:
    - '**/*'
  base-directory: dist
  discard-paths: no
```

---

### ➡️ C3 — Create AWS CodePipeline

1. AWS Console → **CodePipeline** → **Create pipeline**.
2. Name: `reactapp-cicd-demo` | Service role: new.
3. **Source stage:**
   - Provider: **GitHub (Version 2)**
   - Connect GitHub → select repo & branch `main`
   - Detection: **GitHub webhooks**
4. **Build stage:** Provider = **AWS CodeBuild** → **Create project** (below).
5. **Deploy stage:** Provider = **Amazon S3** → bucket `my-react-cicd-demo` → ☑ Extract file.
6. Review → **Create pipeline**.

---

### ➡️ C4 — Create CodeBuild Project

Inside the CodeBuild creation (opened from C3 step 4):

| Setting | Value |
|---------|-------|
| Project name | `react-cicd-pipeline-demo` |
| Environment | Managed image — Amazon Linux 2023 |
| Image | `aws/codebuild/standard:7.0` |
| Service role | New service role |
| Build spec | Use a buildspec file (leave name blank) |

Click **Continue to CodePipeline** → project added to pipeline → **Next**.

The pipeline runs immediately: **Source → Build → Deploy** (~3–5 min).

---

### ➡️ C5 — Configure S3 for Public Static Hosting

After pipeline succeeds and files appear in the bucket:

**Enable Static Website Hosting:**
1. Bucket → **Properties** → **Static website hosting** → **Edit**.
2. Enable → Index document: `index.html` → Error document: `index.html`.
3. **Save changes**.

**Allow Public Access:**
4. **Permissions** → **Block public access** → **Edit** → uncheck all → **Save**.

**Bucket Policy:**
5. **Permissions** → **Bucket policy** → **Edit** → paste:

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Sid": "PublicReadGetObject",
    "Effect": "Allow",
    "Principal": "*",
    "Action": "s3:GetObject",
    "Resource": "arn:aws:s3:::my-react-cicd-demo/*"
  }]
}
```
6. **Save changes**.
7. **Properties → Static website hosting** → copy the **Bucket website endpoint** → open in browser 🎉

---

### ➡️ C6 — Test Full Pipeline (End-to-End)

```bash
# Make a visible change, then:
git add . && git commit -m "test: trigger CI/CD" && git push origin main
```

Watch CodePipeline: Source ✅ → Build ✅ → Deploy ✅ → refresh S3 URL.

---

### ➡️ C7 — (Optional) CloudFront HTTPS CDN

1. **CloudFront** → **Create distribution**.
2. Origin: S3 static website endpoint.
3. Viewer protocol: **Redirect HTTP → HTTPS**.
4. Default root object: `index.html`.
5. **Create** → ~15 min propagation → use `https://xxxxx.cloudfront.net`.

---

### 🗑️ C8 — Clean Up AWS Resources

```
CodePipeline  → Delete: reactapp-cicd-demo
CodeBuild     → Delete: react-cicd-pipeline-demo
S3            → Empty bucket → Delete: my-react-cicd-demo
Azure Foundry → Delete fine-tuned model deployment (avoid idle cost)
```


---
## 📥 Step 9 — Package & Download All Files


In [ ]:
# Step 9.1 — Zip and download the entire project
import shutil, os

shutil.make_archive("finetune_llm_azure_project", "zip", ".", str(BASE))
size = os.path.getsize("finetune_llm_azure_project.zip") / 1024**2
print(f"Zipped: finetune_llm_azure_project.zip ({size:.2f} MB)")


In [ ]:
# Step 9.2 — Trigger download
try:
    from google.colab import files
    files.download("finetune_llm_azure_project.zip")
    print("Download initiated — check browser downloads.")
except ImportError:
    import os
    print("Local run — zip at:", os.path.abspath("finetune_llm_azure_project.zip"))


In [ ]:
# Step 9.3 — Final summary
import os

print("=" * 60)
print("  FINAL PROJECT SUMMARY")
print("=" * 60)
print(f"  Training examples   : {len(train_examples)}")
print(f"  Validation examples : {len(val_examples)}")
print()
print("  Source files written:")
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if d != "node_modules"]
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), BASE)
        print(f"    {rel}")
print()
print("  Next Steps:")
print("    1. Upload JSONL → Azure AI Foundry → Fine-tune GPT-4o")
print("    2. Note endpoint + API key → fill AZURE_* vars above")
print("    3. npm install && npm run dev  (local test)")
print("    4. git push → CodePipeline auto-deploys to S3")
print("=" * 60)
